# Housing Data: Loading and First Cleaning Stage
**By:** Mohamed Issa

**Dataset:** [DOB NOW: Build – Job Application Filings](https://data.cityofnewyork.us/d/w9ak-ipjd)

**Loading method:** the notebook downloads the data from the NYC Open Data API in pages of 25,000 rows, keeping only filings from 2022-01-01 onward (see "Scope: why 2022 onward"). Inside that window it is the full dataset, not a sample.

**How to run:** open the repo folder in VS Code, select the `.venv` Python kernel (packages from `requirements.txt`), and choose Run All. The first run downloads the data to `data/housing_raw.csv`; later runs reuse that file. The output is `data/housing_step1.csv.gz`. The `data/` folder is git-ignored, so neither file is committed.

In [1]:
from pathlib import Path
import io
import time

import pandas as pd
import requests

# All paths are relative to the repo folder. data/ is git-ignored, so nothing in it is committed.
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
RAW = DATA_DIR / "housing_raw.csv"          # raw download (2022 onward), reused on later runs
STEP1 = DATA_DIR / "housing_step1.csv.gz"   # output of this notebook, read by the next cleaning stage

API_URL = "https://data.cityofnewyork.us/resource/w9ak-ipjd.csv"
SCOPE_START = "2022-01-01T00:00:00"
SCOPE_WHERE = f"filing_date >= '{SCOPE_START}'"


def get_csv(params, tries=5):
    """Ask the NYC Open Data API for CSV rows. requests encodes the spaces and quotes in params.
    The server sometimes drops a connection mid-download, so each request is retried a few times."""
    for attempt in range(1, tries + 1):
        try:
            response = requests.get(API_URL, params=params, timeout=300)
            response.raise_for_status()
            return pd.read_csv(io.StringIO(response.text), dtype=str)
        except requests.RequestException as error:
            if attempt == tries:
                raise
            print(f"Request failed ({type(error).__name__}), retry {attempt} of {tries - 1} in {15 * attempt} s")
            time.sleep(15 * attempt)


# Look at a small sample first: size, column names, and what filing_date looks like
df_sample = get_csv({"$limit": 1000})
print(df_sample.shape)
print(df_sample.columns.tolist())
print(df_sample["filing_date"].head())

(1000, 95)
['job_filing_number', 'filing_status', 'house_no', 'street_name', 'borough', 'block', 'lot', 'bin', 'commmunity_board', 'work_on_floor', 'apt_condo_no_s', 'applicant_professional_title', 'applicant_license', 'applicant_first_name', 'applicants_middle_initial', 'applicant_last_name', 'owner_s_business_name', 'applicant_street_name', 'city', 'state', 'zip', 'filing_representative_first_name', 'filing_representative_middle_initial', 'filing_representative_last_name', 'filing_representative_business_name', 'filing_representative_street_name', 'filing_representative_city', 'filing_representative_state', 'filing_representative_zip', 'sprinkler_work_type', 'plumbing_work_type', 'standpipe', 'antenna', 'sign', 'curb_cut', 'fence', 'scaffold', 'shed', 'initial_cost', 'total_construction_floor_area', 'review_building_code', 'little_e', 'unmapped_cco_street', 'request_legalization', 'includes_permanent_removal', 'in_compliance_with_nycecc', 'exempt_from_nycecc', 'building_type', 'exist

In [2]:
# What type does the API store filing_date as? If it were text, ">=" would compare text, not dates.
metadata = requests.get("https://data.cityofnewyork.us/api/views/w9ak-ipjd.json", timeout=60).json()
column_types = {column["fieldName"]: column["dataTypeName"] for column in metadata["columns"]}
print("filing_date type in the API:", column_types["filing_date"])

# Row counts from the API, before and after the 2022 filter
rows_all = int(get_csv({"$select": "count(*)"}).iloc[0, 0])
rows_scope = int(get_csv({"$select": "count(*)", "$where": SCOPE_WHERE}).iloc[0, 0])
# The same filter written a second way (by year), as a cross-check
rows_scope_check = int(get_csv({"$select": "count(*)", "$where": "date_extract_y(filing_date) >= 2022"}).iloc[0, 0])
rows_no_date = int(get_csv({"$select": "count(*)", "$where": "filing_date IS NULL"}).iloc[0, 0])

print("Rows before the 2022 filter:", f"{rows_all:,}")
print("Rows from 2022-01-01 onward:", f"{rows_scope:,}", f"({rows_scope / rows_all:.1%})")
print("Rows left out by the filter: ", f"{rows_all - rows_scope:,}")
print("Same filter, written by year:", f"{rows_scope_check:,}")
print("Rows with no filing_date:   ", f"{rows_no_date:,}")
print("Rows filed before 2022:     ", f"{rows_all - rows_scope - rows_no_date:,}")

filing_date type in the API: calendar_date


Rows before the 2022 filter: 977,813
Rows from 2022-01-01 onward: 731,245 (74.8%)
Rows left out by the filter:  246,568
Same filter, written by year: 731,245
Rows with no filing_date:    2,957
Rows filed before 2022:      243,611


## Scope: why 2022 onward

**Rule:** keep only filings with a `filing_date` on or after 2022-01-01. The filter goes into the API request (`$where`), and the loading cell applies it a second time after reading the file.

**Why:** the team analyses 2022 onward, a recent window where both the housing and the collisions datasets are complete and comparable. The collisions notebook uses the same start date.

**Checking the filter before trusting it:** the API stores `filing_date` as `calendar_date` (printed above), so `>=` compares dates, not text. The same filter written a second way, by year, returns the same count.

**Row counts (printed above):**
- 977,813 rows before the 2022 filter.
- 731,245 rows (74.8%) from 2022-01-01 onward.
- 246,568 rows left out: 243,611 filed before 2022 and 2,957 with no filing date. A row with no date could never be placed in a year anyway.

**2026 is a partial year:** the newest filing date in the download is 2026-10-31 (printed in the loading cell), because the data stops at the download date. 2026 covers January to October only, so its totals must not be compared directly with full years. The dataset is live, so a later run can give slightly different counts.

## Loading the data

**What:** download every filing from 2022 onward, with all 95 columns, into `data/housing_raw.csv`.

**How:**
- The API sends pages of 25,000 rows, sorted by its row id (`:id`) so pages never overlap or skip rows.
- Four pages download at a time, because each request is slow. A failed request is retried.
- Each page is saved to `data/housing_pages/` as soon as it arrives, so a stopped download resumes instead of starting over.
- The final file is written to a temporary file and then renamed, so an interrupted run can't leave a partial `housing_raw.csv`.
- If `data/housing_raw.csv` already exists, the download is skipped. The cell prints which branch ran.

**Safety filter:** after loading, the cell applies the same `>= 2022-01-01` filter again with `pd.to_datetime`. That way an older full download left in `data/` can never reach the output. The cell prints how many rows this removed and compares the row count with the API count.

**Alternatives rejected:**
- Downloading all 977,813 rows and filtering locally: more data for the same result. That fallback is only needed if the API filter doesn't work, and it does.
- Downloading only the columns we keep: faster, but the missing-value table below needs all 95 columns to justify which ones we drop.

In [3]:
from concurrent.futures import ThreadPoolExecutor

PAGE = 25_000
PAGES_DIR = DATA_DIR / "housing_pages"   # one file per page, so a stopped download can resume


def download_page(offset):
    """Download one page of rows (2022 onward) and save it, unless it is already saved."""
    path = PAGES_DIR / f"offset_{offset:07d}.csv"
    if not path.exists():
        # A fixed sort order (:id) makes sure the pages don't overlap or skip rows
        page = get_csv({"$where": SCOPE_WHERE, "$order": ":id", "$limit": PAGE, "$offset": offset})
        tmp = path.with_name(path.name + ".tmp")
        page.to_csv(tmp, index=False)
        tmp.replace(path)
    return path


def download_scope():
    """Download every filing from 2022-01-01 onward, 4 pages at a time (the API is slow per request)."""
    PAGES_DIR.mkdir(exist_ok=True)
    offsets = list(range(0, rows_scope, PAGE))
    with ThreadPoolExecutor(max_workers=4) as pool:
        paths = list(pool.map(download_page, offsets))
    # If the dataset grew since the count above, keep going until a page comes back short
    while len(pd.read_csv(paths[-1], dtype=str)) == PAGE:
        paths.append(download_page(offsets[-1] + PAGE * (len(paths) - len(offsets) + 1)))
    print("Pages downloaded:", len(paths))
    return pd.concat([pd.read_csv(path, dtype=str) for path in paths], ignore_index=True)


if RAW.exists():
    print("Load branch: data/housing_raw.csv already exists, so the download was skipped")
else:
    print("Load branch: data/housing_raw.csv not found, so it was downloaded from the API")
    # Write to a temporary file, then rename, so a stopped download never leaves a partial housing_raw.csv
    tmp = RAW.with_name(RAW.name + ".tmp")
    download_scope().to_csv(tmp, index=False)
    tmp.replace(RAW)
    for path in PAGES_DIR.glob("*"):   # the page files are no longer needed
        path.unlink()
    PAGES_DIR.rmdir()

df_housing = pd.read_csv(RAW, dtype=str)
print("Rows in data/housing_raw.csv:", f"{len(df_housing):,}", "| columns:", df_housing.shape[1])

# Apply the same filter again after loading, so an older full download can never slip into the output
filing_dates = pd.to_datetime(df_housing["filing_date"], format="mixed", errors="coerce")
in_scope = filing_dates >= "2022-01-01"
print("Rows removed by the filter after loading:", f"{(~in_scope).sum():,}")
df_housing = df_housing[in_scope].reset_index(drop=True)
filing_dates = filing_dates[in_scope].reset_index(drop=True)

print("Rows kept:", f"{len(df_housing):,}", "| API count for the same filter:", f"{rows_scope:,}")
print("Earliest filing_date:", filing_dates.min())
print("Latest filing_date:  ", filing_dates.max())
print("Rows per filing year:")
print(filing_dates.dt.year.value_counts().sort_index().to_string())

Load branch: data/housing_raw.csv already exists, so the download was skipped


Rows in data/housing_raw.csv: 731,245 | columns: 95
Rows removed by the filter after loading: 0
Rows kept: 731,245 | API count for the same filter: 731,245
Earliest filing_date: 2022-01-01 00:00:00
Latest filing_date:   2026-10-31 00:00:00
Rows per filing year:
filing_date
2022    141967
2023    141864
2024    155967
2025    158921
2026    132526


## Missing values: inspection

The next three cells measure missing values before any cleaning decision is made:
1. the share missing in each column;
2. why the two stage dates (`approved_date`, `first_permit_date`) are empty;
3. which of the two ZIP columns to use.

In [4]:
# Share of missing values in each column, highest first
missing_pct = df_housing.isna().mean().sort_values(ascending=False) * 100
print(df_housing.shape)
print(missing_pct.round(1).to_string())

(731245, 95)
request_legalization                        100.0
includes_permanent_removal                  100.0
apt_condo_no_s                               99.3
filing_representative_middle_initial         93.0
built_1_information_value                    91.3
built_2_b_information_value                  91.3
built_2_a_information_value                  91.3
built_2_information_value                    91.3
applicants_middle_initial                    69.3
signoff_date                                 66.1
existing_height                              63.8
proposed_height                              58.2
existing_stories                             54.1
proposed_no_of_stories                       47.6
first_permit_date                            44.1
progressinspectionrequirement                39.8
special_inspection_agency_number             39.1
unmapped_cco_street                          36.3
specialinspectionrequirement                 31.8
filing_representative_street_name    

In [5]:
# Why are approved_date and first_permit_date empty? Check the missing share for each filing status
date_check = (df_housing.groupby("filing_status")[["approved_date", "first_permit_date"]]
                        .agg(lambda column: column.isna().mean() * 100)
                        .round(1))
date_check["rows"] = df_housing["filing_status"].value_counts()
print(date_check.sort_values("rows", ascending=False).to_string())

# Rows that reached the permit stage but have no approved date
permit_stage = df_housing["filing_status"].isin(["LOC Issued", "Permit Entire"])
permit_no_approved = permit_stage & df_housing["first_permit_date"].notna() & df_housing["approved_date"].isna()
print("\nLOC Issued or Permit Entire rows:", f"{permit_stage.sum():,}")
print("...of which have a permit date but no approved_date:", f"{permit_no_approved.sum():,}")
print("filing_date missing:", df_housing["filing_date"].isna().sum())

                                                           approved_date  first_permit_date    rows
filing_status                                                                                      
Approved                                                             0.0               99.8  276455
LOC Issued                                                          13.5                0.0  242643
Permit Entire                                                       12.8                0.0  155820
Objections                                                         100.0              100.0   19676
Filing Withdrawn                                                    48.1               88.2    8924
TA Certificate of Operation Issued                                   0.0              100.0    6617
Plan Examiner Review                                               100.0              100.0    6578
CO Issued                                                            0.0                0.0    4646


In [6]:
# There are two ZIP columns, postcode and zip. Which one looks like the building's location?
nyc_prefixes = ("100", "101", "102", "103", "104", "110", "111", "112", "113", "114", "116")
for column in ["postcode", "zip"]:
    share = df_housing[column].str.startswith(nyc_prefixes, na=False).mean() * 100
    missing = df_housing[column].isna().mean() * 100
    print(column, "with an NYC-style ZIP:", round(share, 1), "% | missing:", round(missing, 2), "%")

postcode with an NYC-style ZIP: 99.9 % | missing: 0.09 %


zip with an NYC-style ZIP: 70.8 % | missing: 0.07 %


## Which ZIP column and why

The data has two ZIP columns (printed above):
- `postcode`: 99.9% NYC-style ZIPs, 0.09% missing.
- `zip`: only 70.8% NYC-style ZIPs. It sits in the applicant address block (after `applicant_street_name`, `city` and `state` in the column list printed at the top), so it is the applicant's address, not the building's.

**Choice:** `postcode`, renamed to `zip_code`. It records where the building is, which is what the ZIP × year join needs. `zip` is dropped along with the other applicant details.

After the rename, a check cell asserts that every `zip_code` is a 5-digit string.

## Handling missing values

**Columns removed:**
- `request_legalization` and `includes_permanent_removal` are 100% empty.
- Applicant, owner, and filing representative details don't help answer our question, and they include personal information.
- The `built_*` columns are 91.3% empty. `existing_stories` (54.1%) and `proposed_no_of_stories` (47.6%) are largely empty too, and `total_construction_floor_area` (7.4% empty) is a more complete measure of size.
- Other permit and administrative columns aren't needed for our research questions.

**Columns kept despite missing values (left as missing, not filled):**
- `first_permit_date` (44.1% empty): usually empty because the job hasn't reached the permit stage yet. It is 99.8% empty for `Approved` filings and 100% empty for `Objections`, but 0.0% empty for `LOC Issued`, `Permit Entire`, and `Permit Issued`.
- `approved_date` (11.9% empty) is less reliable. 52,784 of the 398,463 `LOC Issued` and `Permit Entire` rows have a permit date but no approved date, even though a permit comes after approval. So `filing_date` (0 missing) is our time measure, not `approved_date`.
- `existing_dwelling_units` (15.4%), `proposed_dwelling_units` (8.5%), and `total_construction_floor_area` (7.4%) stay missing. Filling them with an average would invent data.

**Rows removed (next cell):** rows missing `job_filing_number`, `filing_date`, or `postcode`. In this window only `postcode` is ever missing, in 654 rows (0.09%). A filing without a ZIP can't be placed in a ZIP × year cell, so it could never enter the join. No rows are missing `filing_date`, because the 2022 filter already left out the 2,957 rows without one.

**Alternative rejected:** filling `postcode` from `zip` or from the coordinates. `zip` is the applicant's address, and turning coordinates into ZIPs needs a separate ZIP boundary dataset. For 0.09% of rows, that isn't worth the risk of assigning a wrong ZIP.

**Also in the next cell:**
- Spaces are trimmed and blank text is treated as missing.
- The source column `commmunity_board` (spelled with three m's in the API, see the column list at the top) is renamed to `community_board`, to follow the snake_case naming rule.

In [7]:
keep_columns = [
    "job_filing_number", "filing_status", "job_type", "building_type",
    "filing_date", "approved_date", "first_permit_date", "current_status_date",
    "borough", "postcode", "latitude", "longitude", "nta", "census_tract",
    "council_district", "commmunity_board", "bbl",
    "initial_cost", "total_construction_floor_area",
    "existing_dwelling_units", "proposed_dwelling_units",
    "full_demolition_work_type_", "job_description",
]

df_clean = df_housing[keep_columns].copy()

# The source column name has a typo (three m's); rename it to snake_case community_board
df_clean = df_clean.rename(columns={"commmunity_board": "community_board"})

# Trim spaces and treat blank text as missing
df_clean = df_clean.apply(lambda column: column.str.strip()).replace("", pd.NA)

# Remove rows we can't identify or place in a ZIP code and year
required = ["job_filing_number", "filing_date", "postcode"]
print("Missing values in each required column:")
print(df_clean[required].isna().sum().to_string())

rows_before = len(df_clean)
df_clean = df_clean.dropna(subset=required)
rows_removed = rows_before - len(df_clean)

print("Rows removed for missing values:", f"{rows_removed:,}")
print("Percent removed:", round(rows_removed / rows_before * 100, 2))
print("Rows left:", f"{len(df_clean):,}")

Missing values in each required column:
job_filing_number      0
filing_date            0
postcode             654
Rows removed for missing values: 654
Percent removed: 0.09
Rows left: 730,591


## Handling duplicates

**Step 1: exact duplicate rows.** Rows that are identical in every kept column add no information, so they are removed. The cell below prints the count: 28 rows. Keeping them was rejected because it would count the same filing twice.

The same cell also counts filing numbers that still repeat (121), and shows the filing suffixes and job types used in the next steps.

In [8]:
exact_duplicates = df_clean.duplicated().sum()
print("Exact duplicate rows:", exact_duplicates)
df_clean = df_clean.drop_duplicates()
print("Rows left:", f"{len(df_clean):,}")

print("Repeated filing numbers:", df_clean["job_filing_number"].duplicated().sum())
print(df_clean["job_filing_number"].head(10).to_string())

filing_suffix = df_clean["job_filing_number"].str.extract(r"-([^-]+)$")[0]
print(filing_suffix.value_counts(dropna=False).head(15).to_string())
print(df_clean["job_type"].value_counts(dropna=False).to_string())

Exact duplicate rows: 28


Rows left: 730,563
Repeated filing numbers: 121
0    B00244768-P3
1    B00464221-S2
2    B00451661-P2
3    B00461532-P2
4    B00360987-P2
5    B00365528-P2
6    B00506606-P2
7    B00458348-P1
8    B00440698-P2
9    B00482269-S8


0
I1    375284
P1    118633
S1     56261
P2     49488
S2     26710
P3     23982
S3     15051
P4     12945
S4      8572
P5      7656
S5      5614
P6      4924
S6      3965
P7      3319
S7      2721
job_type
Alteration                                                589717
New Building                                               55176
Alteration CO                                              49316
ALT-CO - New Building with Existing Elements to Remain     16132
No Work                                                    12233
Full Demolition                                             7989


**Step 2: repeated filing numbers.** After step 1, 121 filing numbers still appear more than once. Their rows mostly differ in `filing_status`, the dates, or the cost (printed below), which suggests they are updates to the same filing. We keep the row with the latest `current_status_date` and remove the 121 older versions. Rows whose date can't be read are sorted first (`na_position="first"`), so they can never be kept as the newest; this run has 0 such dates.

**Alternatives rejected:**
- Keeping the first row: it could carry an outdated status.
- Dropping every version: that would lose 121 real filings.

In [9]:
print("Sample dates:", df_clean["current_status_date"].head(3).tolist())

# Temporary date column, used only to find the newest version of each filing
df_clean["status_date_temp"] = pd.to_datetime(df_clean["current_status_date"], format="mixed", errors="coerce")
print("Dates that couldn't be read:", df_clean["status_date_temp"].isna().sum())

# See which columns differ between the repeated rows
repeated = df_clean[df_clean["job_filing_number"].duplicated(keep=False)]
differences = repeated.groupby("job_filing_number").nunique(dropna=False) > 1
print(differences.sum().sort_values(ascending=False).head(8).to_string())

# Keep the newest version of each filing
rows_before = len(df_clean)
df_clean = (df_clean.sort_values("status_date_temp", na_position="first")
                    .drop_duplicates("job_filing_number", keep="last")
                    .drop(columns="status_date_temp"))

print("Older versions removed:", rows_before - len(df_clean))
print("Repeated filing numbers now:", df_clean["job_filing_number"].duplicated().sum())
print("Rows left:", f"{len(df_clean):,}")

Sample dates: ['2022-06-02T17:28:54.000', '2025-09-12T19:20:57.000', '2024-01-03T04:15:25.000']
Dates that couldn't be read: 0
filing_status                    70
approved_date                    31
current_status_date              31
status_date_temp                 31
filing_date                      31
initial_cost                     24
job_description                  23
total_construction_floor_area     9


Older versions removed: 121
Repeated filing numbers now: 0
Rows left: 730,442


**Step 3: filings versus jobs.** One job can have several filings: the initial filing `I1`, then `P` and `S` filings (the suffix counts are printed above). We keep every filing row and add:
- `job_id`: the job number without its filing suffix.
- `is_initial_filing`: true for `I1` filings.

The same cell renames `postcode` to `zip_code`. Counting `I1` rows gives one row per job and avoids counting a job more than once.

**Effect of the 2022 filter on `I1`:** the filter keeps rows by their own filing date. For a job whose `I1` was filed before 2022, its later `P` and `S` filings from 2022 onward are kept, but its `I1` is not. That is why 29,299 jobs (7.2%) have no `I1` row (printed below). The check cell after it asks the API and finds that 29,000 of these jobs have an `I1` filed before 2022, which leaves 299 with no `I1` anywhere. We did not change the logic.

**For Phase 2:** counting `I1` rows gives jobs that *started* in 2022 or later. Counting `job_id` gives every job with *any* filing in the window. The integration step has to choose one of these on purpose. Also, 15 job IDs have more than one job type, so we use the type on the `I1` row.

In [10]:
df_clean["job_id"] = df_clean["job_filing_number"].str.split("-").str[0]
df_clean["is_initial_filing"] = df_clean["job_filing_number"].str.endswith("-I1")
df_clean = df_clean.rename(columns={"postcode": "zip_code"})

jobs_total = df_clean["job_id"].nunique()
jobs_with_i1 = df_clean.loc[df_clean["is_initial_filing"], "job_id"].nunique()
jobs_without_i1 = jobs_total - jobs_with_i1

print("Unique jobs:", f"{jobs_total:,}")
print("Jobs with an I1 filing:", f"{jobs_with_i1:,}")
print("Jobs without an I1 filing:", f"{jobs_without_i1:,}", f"({jobs_without_i1 / jobs_total:.1%})")
print("Job IDs with more than one job type:", (df_clean.groupby("job_id")["job_type"].nunique() > 1).sum())

Unique jobs: 404,583
Jobs with an I1 filing: 375,284
Jobs without an I1 filing: 29,299 (7.2%)


Job IDs with more than one job type: 15


In [11]:
# Why do some jobs have no I1 row? Ask the API which of them have an I1 filed before 2022.
missing_i1_jobs = set(df_clean.loc[~df_clean["job_id"].isin(
    df_clean.loc[df_clean["is_initial_filing"], "job_id"]), "job_id"])
old_i1 = get_csv({"$select": "job_filing_number", "$limit": 1_000_000,
                  "$where": f"filing_date < '{SCOPE_START}' AND job_filing_number like '%-I1'"})
old_i1_jobs = set(old_i1["job_filing_number"].str.split("-").str[0])
explained = len(missing_i1_jobs & old_i1_jobs)

print("I1 filings dated before 2022 (API):", f"{len(old_i1):,}")
print("Jobs without an I1 row whose I1 was filed before 2022:", f"{explained:,}")
print("Jobs without an I1 row anywhere in the dataset:", f"{len(missing_i1_jobs) - explained:,}")

I1 filings dated before 2022 (API): 195,173
Jobs without an I1 row whose I1 was filed before 2022: 29,000
Jobs without an I1 row anywhere in the dataset: 299


In [12]:
assert df_clean["zip_code"].str.fullmatch(r"\d{5}").all(), "non 5-digit ZIPs present"
print("ZIPs checked:", df_clean["zip_code"].nunique(), "unique, all 5-digit strings")

ZIPs checked: 225 unique, all 5-digit strings


## Saving the output

The next cell saves the table to `data/housing_step1.csv.gz` (git-ignored). It then reloads the file the way the next stage reads it (`dtype={"zip_code": "string"}`) and checks four things:
- every ZIP is still a 5-digit string;
- the earliest filing year is 2022 or later;
- no filing number repeats;
- no required value is missing.

In [13]:
df_clean.to_csv(STEP1, index=False)   # the .gz ending makes pandas compress the file with gzip
print("Saved", STEP1, df_clean.shape)
print(round(STEP1.stat().st_size / 1024**2), "MB")

# Reload the file the same way the next stage reads it, then check it
reloaded = pd.read_csv(STEP1, dtype={"zip_code": "string"}, low_memory=False)
print("Reloaded shape:", reloaded.shape)
print("zip_code type after reload:", reloaded["zip_code"].dtype)
print("All ZIPs are 5-digit strings:", bool(reloaded["zip_code"].str.fullmatch(r"\d{5}").all()))
print("Min filing year:", pd.to_datetime(reloaded["filing_date"], format="mixed").dt.year.min())
print("Repeated filing numbers:", reloaded["job_filing_number"].duplicated().sum())
print("Missing required values:", reloaded[["job_filing_number", "filing_date", "zip_code"]].isna().sum().sum())
print("Columns:", reloaded.columns.tolist())

Saved data/housing_step1.csv.gz (730442, 25)
66 MB


Reloaded shape: (730442, 25)
zip_code type after reload: string
All ZIPs are 5-digit strings: True
Min filing year: 2022
Repeated filing numbers: 0
Missing required values: 0
Columns: ['job_filing_number', 'filing_status', 'job_type', 'building_type', 'filing_date', 'approved_date', 'first_permit_date', 'current_status_date', 'borough', 'zip_code', 'latitude', 'longitude', 'nta', 'census_tract', 'council_district', 'community_board', 'bbl', 'initial_cost', 'total_construction_floor_area', 'existing_dwelling_units', 'proposed_dwelling_units', 'full_demolition_work_type_', 'job_description', 'job_id', 'is_initial_filing']


## Summary of row counts

| Step | Rows removed | Rows left |
|---|---:|---:|
| Rows before the 2022 filter | – | 977,813 |
| Filed before 2022 or no filing date (2022 filter) | 246,568 | 731,245 |
| Missing filing number, date, or ZIP | 654 | 730,591 |
| Exact duplicate rows | 28 | 730,563 |
| Older versions of repeated filing numbers | 121 | 730,442 |

The output `data/housing_step1.csv.gz` has 730,442 filing rows and 25 columns. They represent 404,583 jobs: 375,284 with an `I1` row and 29,299 without one, of which 29,000 have their `I1` filed before 2022 (see step 3).

**Contribution:** Mohamed Issa loaded the housing data, applied the 2022 scope, handled missing values and duplicates, and prepared it for the next cleaning stage.

## Prompts used

Full log in `AI_PROMPTS.txt` (entries by Mohamed Issa).

AI-assisted cells:
- **Claude (claude.ai):** the original cleaning plan (column selection, dropping rows missing a filing number, date, or ZIP, the duplicate handling, `job_id` and `is_initial_filing`), the paged API download, the `na_position="first"` fix, and the ZIP and date checks.
- **Claude Code (in VS Code):** the team lead's fixes. That covers the local `data/` paths in place of Google Drive; the 2022 filter with its API type check and row counts; the resumable 4-at-a-time download with retries; the date-stage, ZIP-column, and pre-2022 `I1` check cells; the `community_board` rename; the ZIP assert; the save-and-reload check; and the rewritten markdown cells, with every number taken from the printed output.